`control` 和 `sympy` 常用于控制系统分析，前者用于数值仿真和时域、频域分析，后者用于符号运算。

- `sp.together` 表示通分运算
- `sp.factor` 表示因式分解
- `sp.cancel` 表示约分并展开为多项式
- `sp.expand` 表示多项式展开
- `sp.apart` 表示部分分式展开
- `sp.fraction` 表示解包分子和分母

`IPython.display` 中的 `Math` 声明其中的变量应渲染为 Latex 公式， `sp.latex` 表示将 `sympy` 中的原始字符串转换为 Latex 字符串。

In [ ]:
# 常见符号运算
import sympy as sp
from IPython.display import Math

s = sp.symbols('s')
G1 = 1/(s**2 + s*2 + 1)
G2 = 1/(s+2)
G3 = G1 + G2

display(Math(rf'G_1+G_2 = \; {sp.latex(G3)}'))
print()

# 通分
display(Math(rf'通分： \; {sp.latex(sp.together(G3))}'))
print()

# 因式分解
display(Math(rf'因式分解： \; {sp.latex(sp.factor(G3))}'))
print()

# 约分
display(Math(rf'约分： \; {sp.latex(sp.cancel(G3))}'))
print()

# 解包分子与分母(需要先合成一个分式)
num,den = sp.fraction(sp.together(G3))
display(Math(rf'分子： \; {sp.latex(num)}'))
display(Math(rf'分母： \; {sp.latex(sp.expand(den))}'))
print()

# 部分分式展开
display(Math(rf'部分分式展开： \; {sp.latex(sp.apart(sp.together(G3),s))}'))

先将给定与系统的传递函数相乘获得输出，然后进行 Laplace 反变换 `sp.inverse_laplace_transform` ，如果只看 $t \geq 0$的部分，可以替换单位阶跃响应 `sp.Heaviside(t)`。

In [ ]:
# 求解一阶系统的响应
import sympy as sp
from IPython.display import Math

s,t = sp.symbols('s t')
T = sp.symbols('T',positive=True)

G = 1 / (T*s + 1)
R = 1 / s**2
C = G * R
C_a = sp.apart(C,s)  # 部分分式分解
c_t = sp.inverse_laplace_transform(C_a,s,t) # Laplace 反变换
c_t_positive = c_t.subs(sp.Heaviside(t),1) # 右半平面阶跃函数为1
c_t_1s = c_t.subs(t,1) # 求t=1时刻的响应
c_t_0s = sp.limit(c_t,t,0,dir='+') # 求t=0+时刻的极限
e_t = t * sp.Heaviside(t) - c_t # 求误差响应

display(Math(rf'C(s) = \; {sp.latex(C)}'))
print()
display(Math(rf'C(s) = \; {sp.latex(C_a)}'))
print()

display(Math(rf'c(t) = \; {sp.latex(c_t)}'))
display(Math(rf'c(t) = \; {sp.latex(c_t_positive)}'))
display(Math(rf'c(1) = \; {sp.latex(c_t_1s)}'))
display(Math(rf'c(0+) = \; {sp.latex(c_t_0s)}'))
print()

display(Math(rf'e(t) = \; {sp.latex(e_t)}'))
display(Math(rf'e(\infty) = \; {sp.latex(sp.limit(e_t,t,sp.oo))}'))

`sp.roots()` 用于求解单变量多项式方程。
- 返回格式：返回一个**字典**，键是根，值是重数；
- 完整性：如果无法求出所有根（例如高于四次且无法因式分解的多项式），它只会返回能求出的根，不会报错，也不会抛出占位符。

`sp.solve()` 通用的代数方程求解器。
- 返回格式：返回一个**列表**；
- 重根处理：如果方程有重根，`solve` 会把重复的根重复列出；
- 完整性：如果无法求出精确解析解，可能会返回 `RootOf` （表示无法显示表达的根）或者空列表。

In [ ]:
# 求二阶系统的响应
import sympy as sp
from IPython.display import Math

s,t,J,B,K = sp.symbols('s t J B K')
omega_n,zeta = sp.symbols('omega_n zeta')

G = K / (J*s**2 + B*s +K)

display(Math(rf'G(s) = \; {sp.latex(G,order='grevlex')}'))
print()

num,den = sp.fraction(G)
poles = sp.roots(den,s)  # sp.roots()输出字典，用于分析极点重数
# poles = sp.solve(den,s) # sp.solve()求解特征方程，只能求出极点，无法求出重数
# poles = sp.solve(sp.Eq(den,0),s) # sp.solve()的输入默认为0，因此两种写法一致
p = [root for root in poles] # 列表推导式和字典的循环遍历

display(Math(rf'二阶系统的极点为： \; {sp.latex(p[0])},{sp.latex(p[1])}'))
display(Math(rf'两个极点对应的重数分别为： \; {sp.latex(poles[p[0]])},{sp.latex(poles[p[1]])}'))
print()

eqs = [
    sp.Eq(K/J,omega_n**2),
    sp.Eq(B/J,2*zeta*omega_n)
]
params = sp.solve(eqs,(K,B),dict=True)[0] # dict=True表示以字典形式存储解，solve实质返回一个字典列表，[0]表示去其中的一组解
G_standard = sp.cancel(G.subs(params))

display(Math(rf'B = \; {sp.latex(params[B])}'))
display(Math(rf'K = \; {sp.latex(params[K])}'))
display(Math(rf'该二阶系统可化为标准二阶系统：\; {sp.latex(G_standard)}'))
print()

num,den = sp.fraction(G_standard)
poles = sp.roots(den,s)

p = [root for root in poles]

display(Math(rf'二阶系统的极点为： \; {sp.latex(sp.simplify(p[0]))},{sp.latex(sp.simplify(p[1]))}'))
display(Math(rf'两个极点对应的重数为： \; {sp.latex(poles[p[0]])},{sp.latex(poles[p[1]])}'))

In [1]:
import sympy as sp
from IPython.display import Math

s,t = sp.symbols('s t')
omega_d,omega_n,zeta = sp.symbols('omega_d omega_n zeta',positive=True)

C = (omega_n**2)/(s*(s**2+2*zeta*omega_n*s+omega_n**2))
display(Math(rf'C(s) = \; {sp.latex(C)}'))
print()

C_apart = sp.apart(C,s)
display(Math(rf'C(s) = \; {sp.latex(C_apart)}'))
print()

c_t = sp.inverse_laplace_transform(C_apart,s,t)
c_t_pos = sp.simplify(c_t.subs(sp.Heaviside(t),1)) # 一般先化简再替换
display(Math(rf'c(t) = \; {sp.latex(c_t_pos.subs(omega_n*sp.sqrt(1-zeta**2),omega_d))} \quad ,t > 0'))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

计算 cascaded、parallel、feedback 三种情况下的传递函数，控制框图如下：

<img src="./assets/chap1/cascaded_parallel_and_feedback_systems.svg" width="600">

In [ ]:
# control用于数值仿真与时域/频域分析
import control as ct

s = ct.tf('s')

G1 = 10 / (s**2 + s*2 + 10)
G2 = 5 / (s + 5)

display(ct.series(G1,G2))
display(ct.parallel(G1,G2))
display(ct.feedback(G1,G2))